# Bag of Words
__MATH 3480__ - Dr. Michael Olson

Reading:
* a


-----
### Bag-of-words example

Imagine you have an assignment that is graded and given feedback. Some examples are as follows:

| Grade | Feedback |
| :---: | :------- |
|   A   | Fantastic work! |
|   A   | Great job! |
|   A   | Perfect! |
|   B   | Good job |
|   B   | Almost perfect |
|   B   | So close |
|   C   | Good, but needs work |
|   C   | Needs some help |
|   D   | Poor work |
|   D   | Try harder next time |

Now, we create a list (bag) of these words and count how frequently they are associated with each grade.

| Word      |   A   |   B   |   C   |   D   |
| :-------- | :---: | :---: | :---: | :---: |
| Fantastic |   1   |   0   |   0   |   0   |
| Work      |   1   |   0   |   1   |   1   |
| Great     |   1   |   0   |   0   |   0   | 
| job       |   1   |   1   |   0   |   0   |
| Perfect   |   1   |   1   |   0   |   0   |
| Good      |   0   |   1   |   1   |   0   |
| Almost    |   0   |   1   |   0   |   0   |
| So        |   0   |   1   |   0   |   0   |
| close     |   0   |   1   |   0   |   0   |
| but       |   0   |   0   |   1   |   0   |
| needs     |   0   |   0   |   2   |   0   |
| some      |   0   |   0   |   1   |   0   |
| help      |   0   |   0   |   1   |   0   |
| Poor      |   0   |   0   |   0   |   1   |
| Try       |   0   |   0   |   0   |   1   |
| harder    |   0   |   0   |   0   |   1   |
| next      |   0   |   0   |   0   |   1   |
| time      |   0   |   0   |   0   |   1   |

How big is this array of words? (Do a google search for "How many words are used in the English language?")
* Over 170,000 words available
* Only 20,000 commonly used
* Only 3,000 really needed in everyday speech

We create an empty array of 20,000 words. This is our bag of words.
$$[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,\dots,0]$$

* Each position represents a word
* First two spots in the array are reserved
    * `array[0] = SOS` where SOS = Start of Sentence
    * `array[1] = EOS` where EOS = End of Sentence
    * `array[n]` is the last element of the array, reserved for any of those special words not in our array.

    $$[SOS, EOS, a, of, the, if, is, did, not, and, me, you, have, get, good, perfect, worth, amazing, fantastic, job, work, \dots$$
    $$\qquad\dots your, mine, ",", "!", ".", know, question, shall, go, let, \dots, (special~words)]$$

* Fill the array with a count of how often those words occur in our example

Note that derivations of the same word are grouped together:
* worthy ==> worth
* questions ==> question

Quick example: On one of your assignments, I write the following feedback:
> Fantastic work, John! Your work is worthy of JPL! Let me know if you have questions. Michael.

Putting this into the bag of words,
$$[4,4,0,1,0,1,1,0,0,0,1,1,1,0,0,0,1,0,1,0,2,\dots$$
$$\qquad 1,0,1,2,2,1,1,0,0,1,\dots, 3]$$

We would do this for all feedback given to all students (1,000's or even 1,000,000's of students), linked with the grade given to each student. This is used to train the model (logistic regression, naive bayes, DNN, etc.). Then, we would take the array from our example and feed it into our newly-trained model, and it should be able to predict the appropriate grade for the student just based on the feedback!

-----
We are now going to create a model using reviews for a restaurant. As we build the model, remember the steps:

<img src="https://github.com/drolsonmi/math3480/blob/main/Notes/Images/3480_NLP_Process.png?raw=true" height=400 alt="The steps of Natural Language Processing">

### Importing the Libraries

In [1]:
import numpy as np
import pandas as pd

### Importing the Dataset

In [2]:
dataset = pd.read_csv('Data/Restaurant_Reviews.tsv', delimiter='\t', quoting=3)

print(dataset.iloc[228])
print(dataset.shape)
dataset.head()

Review    As for the "mains," also uninspired.
Liked                                        0
Name: 228, dtype: object
(1000, 2)


,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [7]:
dataset.loc[3]['Review']

'Stopped by during the late May bank holiday off Rick Steve recommendation and loved it.'

### Cleaning the text

In [4]:
import re
import nltk     # For the stopwords
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

# List that will contain each cleaned review
corpus = []

# Iterate through each review, cleaning then adding to the corpus
for i in range(0, dataset.shape[0]):
  review = dataset['Review'][i]
  review = re.sub(r'[^a-zA-Z!]', ' ', review)
  review = review.lower()
  review = review.split()
  # Stemming
  ps = PorterStemmer()
  all_stopwords = stopwords.words('english')
  all_stopwords.remove('not')
  review = [ps.stem(word) for word in review if not word in set(all_stopwords)]
  review = ' '.join(review)
  corpus.append(review)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\michael.olson2\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [5]:
print(corpus)

['wow love place', 'crust not good', 'not tasti textur nasti', 'stop late may bank holiday rick steve recommend love', 'select menu great price', 'get angri want damn pho', 'honeslti tast fresh', 'potato like rubber could tell made ahead time kept warmer', 'fri great', 'great touch', 'servic prompt', 'would not go back', 'cashier care ever say still end wayyy overpr', 'tri cape cod ravoli chicken cranberri mmmm!', 'disgust pretti sure human hair', 'shock sign indic cash', 'highli recommend', 'waitress littl slow servic', 'place not worth time let alon vega', 'not like', 'burritto blah!', 'food amaz', 'servic also cute', 'could care less interior beauti', 'perform', 'right red velvet cake ohhh stuff good', 'never brought salad ask', 'hole wall great mexican street taco friendli staff', 'took hour get food tabl restaur food luke warm sever run around like total overwhelm', 'worst salmon sashimi', 'also combo like burger fri beer decent deal', 'like final blow!', 'found place accid could 

### Creating the Bag of Words model

In [27]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=500)
X = cv.fit_transform(corpus).toarray()
y = dataset.iloc[:,-1].values

### Cross Validation


In [28]:
X.shape

(1000, 500)

In [29]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state=65)

### Training the model (Naive Bayes)

In [30]:
from sklearn.naive_bayes import GaussianNB
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

GaussianNB()

### Testing the model

In [31]:
y_pred = nb_model.predict(X_test)

### Evaluation

In [32]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[54 47]
 [10 89]]
              precision    recall  f1-score   support

           0       0.84      0.53      0.65       101
           1       0.65      0.90      0.76        99

    accuracy                           0.71       200
   macro avg       0.75      0.72      0.71       200
weighted avg       0.75      0.71      0.71       200



In [33]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# n_estimators :    Number of models to make
# max_samples :     Number of instances in each bag
# bootstrap :       If True, bootstraps with replacement. If False, without replacement
# n_jobs :          Number of processors to use (-1 means use all processors)
# oob_score :       Calculate Out-of-Bag Evaluation
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500, max_samples=100, 
    bootstrap=True, n_jobs=-1, oob_score=True)
    
bag_clf.fit(X_train, y_train)
print(" OOB Score = ",bag_clf.oob_score_)
y_predict = bag_clf.predict(X_test)

from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y_test,y_predict))
from sklearn.metrics import accuracy_score
print(" Accuracy Score = ", accuracy_score(y_test,y_predict))

 OOB Score =  0.7325
[[98  3]
 [56 43]]
 Accuracy Score =  0.705


In [34]:
print(classification_report(y_test,y_predict))

              precision    recall  f1-score   support

           0       0.64      0.97      0.77       101
           1       0.93      0.43      0.59        99

    accuracy                           0.70       200
   macro avg       0.79      0.70      0.68       200
weighted avg       0.78      0.70      0.68       200

